<a href="https://colab.research.google.com/github/ole0246/LanguageModelsAsCognitiveModels/blob/main/DSC_291_HW1_Part_2_Training_BabyLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Overview, Setup and Instructions

## Overview

In this homework, you will train LMs from scratch on synthetic data, perform syntactic filtering and manipulations on naturalistic data, and also make your own mini-research contributions.

1. **Section 1 (0 points):** Setup and Instructions
2. **Section 2 (6 points):** Training a Tiny GPT on Synthetic Data
    - Generate Synthetic Corpus (0 point)
    - Synthetic Test Suite (1 point)
    - Train a Word-Level GPT (1 point)
    - Plot Results (1 point)
    - Mini-Project 1 (3 points)
    - Generative AI Usage (0 points)
3. **Section 3 (7 points):** Controlled Rearing and Syntactic Editing
    - Introduction to Stanza (0 points)
    - Filter Sentences (2 points)
    - Create Counterfactual Corpora (2 points)
    - Mini-Project 2 (5 points)
    - Generative AI Usage (0 points)
5. **Section 4 (0 points):** Convert to .pdf and submit

Note: We've decided to split HW1 up into two parts. Part 1 is worth 12 points. Part 2 will be released later and is worth 13 points.

**GENERATIVE AI POLICY:**
For this homework, the use of LLMs is allowed for brainstorming and code generation BUT you need to do two things:

- You must disclose all generative AI use.
- You must describe all generative AI contributions *in your own words*.

**NOTE:**
Generative AI tools (Claude and Gemini) were used to assist with writing code for this assignment, including code to generate data from a PCFG and code to train small LMs. Generated code was reviewed and tested by the instructor, and code had to be regenerated multiple times following instructions. The contents of the PCFG, evaluation data, and other code was manually written, and the design of the assignment was conceived and developed without the use of AI tools.

## Instructions

- Step 1: Make a copy of this notebook.
- Step 2: Connect to GPU (if needed).
   - We recommend connecting to T4 GPUs for compute-intensive tasks, like training or evaluating small LMs. To do so, go to `Runtime > Change runtime type > Hardware accelerator` and select `T4 GPU`. There are usage limits (for free accounts), so you may want to connect to GPU only when needed. This notebook can also be run on university clusters where free GPUs are provided (let us know if you need support with this).
- Step 3: Run the setup cells.
- Step 4: Complete the assignment.
- Step 5: Convert the notebook to .pdf (see instructions at the bottom).
- Step 6: Submit using Gradescope (link on the course canvas site).

## Setup

In [ ]:
!pip install -q datasets transformers huggingface_hub

In [ ]:
import math
import time
import random
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
torch.manual_seed(1337)
random.seed(1337)


# Section 2: Training a Tiny GPT on Synthetic Data (6 points)

## Generate a Synthetic Corpus from a Grammar (0 points)

The following codeblock defines a class `PCFG` for a probabilistic context free grammar. You only need to understand a few things about this code:

1. You initialize a `PCFG` by passing in a `.csv` specifying the grammar.
```python
    grammar_file = 'PCFG - My Grammar.csv'
    pcfg = PCFG(grammar_file)
```

2. The grammar `.csv` has the following columns:

    - weight - the weight with which `lhs` explands to `rhs`, relative to other rules with the same `lhs`.
    - lhs - the left-hand side of the rule, i.e., `S` in the rule `S -> NP VP`.
    - rhs - the right-hand side of the rule.
    - head_index  : 1-based index of the head child in rhs (0 = terminal/no children)
    - dep_labels  : space-separated list of dependency labels, one per RHS symbol,
                        parallel to rhs. Use '_' for the head position.

    e.g.,
    
    | weight | lhs | rhs | head_index | dep_labels |
    |---|---|---|---|---|
    | 1 | ROOT | S <EOS> | 2 | root |
    | 1 | S | NP_a VP | 2 | nsubj |
    | 10 | VP | V_i | 1 | _ |
    | 20 | VP | V_t NP_a | 1 | dobj |
    | 4 | VP | V_c CP | 1 | ccomp |
    | 4 | VP | V_p PP | 1 | obl |
    | 4 | VP | V_to VP_to | 1 | xcomp |
    | 0.1 | S | [ S ] | 2 | punc punc |


3. You can sample sentences from the `PCFG` in several modes:

    - 'plain'     - bare word string, no annotation
    - 'bracketed' - Penn-style brackets with POS/constituent labels
    - 'dep'       - brackets annotated with POS, dep label, HEAD_TERMINAL_IDX,
                        and TERMINAL_IDX (-1 for non-terminals)
    - 'conllu'    - CSV with columns: IDX, FORM, POS, HEAD_TERMINAL_IDX, DEP_LABEL

### Define PCFG Class

You can hide this long codeblock after running, since you don't need to do anything here.

In [ ]:
import random
import csv
import io


class PCFG:
    def __init__(self, grammar_file):
        self.rules = None
        self.sep = ","
        self.load_rules(grammar_file)

    # Grammar loading
    def load_rules(self, grammar_file):
        new_rules = {}
        with open(grammar_file, 'r') as g_file:
            lines = g_file.readlines()
        for l in lines:
            if l.startswith(('#', ' ', self.sep, '\n')) or len(l) < 1:
                continue
            if l.find('#') != -1:
                l = l[:l.find('#')]
            parts = l.rstrip().split(self.sep)
            if len(parts) < 3:
                continue
            weight     = float(parts[0])
            lhs        = parts[1]
            rhs        = parts[2]
            head_index = int(parts[3]) if len(parts) > 3 and parts[3].strip() else 0
            raw_dep = parts[4].strip() if len(parts) > 4 else ''
            dep_labels = raw_dep.split() if raw_dep else []

            if lhs not in new_rules:
                new_rules[lhs] = []
            new_rules[lhs].append([rhs, weight, head_index, dep_labels])

        # Normalise weights
        for lhs, poss in new_rules.items():
            total = sum(r[1] for r in poss)
            for r in poss:
                r[1] /= total

        self.rules = new_rules

    # Public entry point
    def sample_sentence(self, max_expansions=200, mode='plain'):
        self._term_counter = [0]

        # Phase 1: build tree, computing each node's own lexical head (bottom-up)
        tree = self._expand_symbol('ROOT', parent_dep='_', max_depth=max_expansions)

        # Phase 2: annotate dep labels and POS top-down
        self._annotate_dep(tree, inherited_dep='_')

        # Phase 3: assign gov_idx (= parent's head_idx) top-down; root gets -1
        self._annotate_gov(tree, parent_head_idx=-1)

        if mode == 'plain':
            return self._render_plain(tree)
        elif mode == 'bracketed':
            return self._render_bracketed(tree)
        elif mode == 'dep':
            return self._render_dep(tree)
        elif mode == 'conllu':
            return self._render_conllu(tree)
        else:
            raise ValueError(f"Unknown mode '{mode}'. Choose from: plain, bracketed, dep, conllu")

    # Phase 1: tree expansion (bottom-up head_idx)
    def _expand_symbol(self, symbol, parent_dep, depth=0, max_depth=200):
        node = {
            'symbol':   symbol,
            'pos':      None,
            'dep':      parent_dep,
            'head_idx': None,   # own lexical head, bubbled up
            'gov_idx':  None,   # governing word, pushed down
            'term_idx': None,
            'word':     None,
            'children': [],
        }

        if symbol not in self.rules or depth > max_depth:
            # Terminal
            self._term_counter[0] += 1
            node['word']     = symbol
            node['term_idx'] = self._term_counter[0]
            node['head_idx'] = self._term_counter[0]
            return node

        rhs_str, _, head_index, dep_labels = self._sample_rule(symbol)
        rhs_symbols = rhs_str.split()

        for i, child_sym in enumerate(rhs_symbols):
            is_head_child = (i + 1 == head_index)  # 1-based
            if is_head_child or head_index == 0:
                child_dep = ''
            else:
                n_nonhead = len(rhs_symbols) - 1
                if n_nonhead == 0 or not dep_labels:
                    child_dep = self._get_child_dep(child_sym)
                elif len(dep_labels) == len(rhs_symbols):
                    child_dep = dep_labels[i] if dep_labels[i] != '_' else ''
                elif len(dep_labels) == n_nonhead:
                    nonhead_idx = i if i < (head_index - 1) else i - 1
                    child_dep = dep_labels[nonhead_idx]
                elif len(dep_labels) == 1:
                    child_dep = dep_labels[0]
                else:
                    child_dep = self._get_child_dep(child_sym)
            child_node = self._expand_symbol(child_sym, parent_dep=child_dep,
                                             depth=depth + 1, max_depth=max_depth)
            node['children'].append(child_node)

        if head_index > 0 and head_index <= len(node['children']):
            node['head_idx'] = node['children'][head_index - 1]['head_idx']
        elif node['children']:
            node['head_idx'] = node['children'][0]['head_idx']

        return node

    # Phase 2: annotate dep labels and POS (top-down)
    def _annotate_dep(self, node, inherited_dep):
        if not node['dep']:
            node['dep'] = inherited_dep

        if node['word'] is not None:
            return

        parent_head_idx = node['head_idx']

        for child in node['children']:
            if child['word'] is not None:
                child['pos'] = node['symbol']

            if child['head_idx'] == parent_head_idx:
                child_inherited = node['dep']
            else:
                child_inherited = child['dep'] or node['dep']

            self._annotate_dep(child, inherited_dep=child_inherited)

    # Phase 3: assign gov_idx (top-down)
    def _annotate_gov(self, node, parent_head_idx):
        node['gov_idx'] = parent_head_idx

        if node['word'] is not None:
            return

        this_head_idx = node['head_idx']

        for child in node['children']:
            if child['head_idx'] == this_head_idx:
                self._annotate_gov(child, parent_head_idx=parent_head_idx)
            else:
                self._annotate_gov(child, parent_head_idx=this_head_idx)

    # Helpers
    def _get_child_dep(self, symbol):
        if symbol in self.rules:
            labels = self.rules[symbol][0][3]
            return next((l for l in labels if l != '_'), '')
        return ''

    def _sample_rule(self, symbol):
        poss       = self.rules[symbol]
        cumulative = 0.0
        sample     = random.random()
        for rule in poss:
            cumulative += rule[1]
            if sample <= cumulative:
                return rule
        return poss[-1]

    def _is_preterminal(self, node):
        return (node['word'] is None
                and len(node['children']) == 1
                and node['children'][0]['word'] is not None)

    # Output renderers
    def _render_plain(self, node):
        if node['word'] is not None:
            return node['word']
        return ' '.join(self._render_plain(c) for c in node['children'])

    def _render_bracketed(self, node):
        if node['word'] is not None:
            return node['word']
        if self._is_preterminal(node):
            return f"({node['symbol']} {node['children'][0]['word']})"
        children_str = ' '.join(self._render_bracketed(c) for c in node['children'])
        return f"({node['symbol']} {children_str})"

    def _render_dep(self, node):
        dep     = node['dep'] or '_'
        gov_idx = node['gov_idx'] if node['gov_idx'] is not None else -1

        if node['word'] is not None:
            pos = node['pos'] or node['symbol']
            return f"({pos} {dep} {gov_idx} {node['term_idx']} {node['word']})"

        if self._is_preterminal(node):
            child = node['children'][0]
            child_dep = child['dep'] or dep
            child_gov = child['gov_idx'] if child['gov_idx'] is not None else gov_idx
            return f"({node['symbol']} {child_dep} {child_gov} {child['term_idx']} {child['word']})"

        children_str = ' '.join(self._render_dep(c) for c in node['children'])
        return f"({node['symbol']} {dep} {gov_idx} -1 {children_str})"

    def _render_conllu(self, node):
        """CSV: IDX, FORM, POS, HEAD_TERMINAL_IDX, DEP_LABEL"""
        rows = self._collect_terminal_rows(node)
        output = io.StringIO()
        writer = csv.writer(output)
        writer.writerow(['IDX', 'FORM', 'POS', 'HEAD_TERMINAL_IDX', 'DEP_LABEL'])
        writer.writerows(rows)
        return output.getvalue()

    def _collect_terminal_rows(self, node):
        if self._is_preterminal(node):
            child     = node['children'][0]
            dep       = child['dep'] or node['dep'] or '_'
            gov_idx   = child['gov_idx'] if child['gov_idx'] is not None else -1
            return [[child['term_idx'], child['word'], node['symbol'], gov_idx, dep]]
        if node['word'] is not None:
            pos     = node['pos'] or node['symbol']
            gov_idx = node['gov_idx'] if node['gov_idx'] is not None else -1
            return [[node['term_idx'], node['word'], pos, gov_idx, node['dep'] or '_']]
        rows = []
        for child in node['children']:
            rows.extend(self._collect_terminal_rows(child))
        return rows

### Load Grammar

Load the grammar

1. Download the grammar from the course drive: https://drive.google.com/file/d/1AnyOlKTvHNrFvuY0mumXRuAS0RR1WXfe/view?usp=drive_link
2. Drag and drop the grammar `.csv` into the file system for your runtime (click on the folder icon on the left panel of the notebook).
3. Run the following codeblock several times to load the grammar inspect what the generated sentences look like.
4. Generate train and dev sets.

In [ ]:
# Load the PCFG and test outputs
grammar_file = 'PCFG.csv'
pcfg = PCFG(grammar_file)

print("=== plain ===")
print(pcfg.sample_sentence(mode='plain'))

print("\n=== bracketed ===")
print(pcfg.sample_sentence(mode='bracketed'))

print("\n=== dep ===")
print(pcfg.sample_sentence(mode='dep'))

print("\n=== conllu ===")
print(pcfg.sample_sentence(mode='conllu'))

In [ ]:
# Generate train and dev sets

TRAIN_SIZE = 10**5
DEV_SIZE = 10**4

for split in ["train", "dev"]:
    sents = []
    n = TRAIN_SIZE if split == "train" else DEV_SIZE
    for i in tqdm(range(n)):
        sents.append(pcfg.sample_sentence(mode='plain'))

    with open(f"{split}.txt", "w") as f:
        for sent in sents:
            f.write(f"{sent}\n")

## Synthetic Test Suite (1 point)

Add the test suite

1. Download the minimal pairs test eval dataset from the following link: https://drive.google.com/file/d/16PA_A_lMZegbIaAhic4_qiW0rCw8NrV5/view?usp=drive_link
2. Drag and drop the eval dataset into the notebook file system.
3. Describe two of the test suites (there are 10 to choose from).
4. Add a new test suite covering a phenomenon not included in the eval dataset.
5. Load the eval dataset including your new test suite.

Describe **two** of the test suites:

\<YOUR ANSWER\>

Describe your new test suite:

\<YOUR ANSWER\>

In [ ]:
# Load eval data and create your own test suite

eval_df = pd.read_csv('eval.csv', index_col=0)

my_test_suite = [
    {"test_suite": "TODO",
     "pair_id": 1,
     "grammatical": "TODO",
     "ungrammatical": "TODO"},
    # Create 10 minimal pairs
    # <YOUR ANSWER>
]

eval_df = pd.concat([eval_df, pd.DataFrame(my_test_suite)])

syn_test_cases = {
    suite: list(zip(g.tolist(), b.tolist()))
    for suite, sub in eval_df.groupby('test_suite')
    for g, b in [(sub['grammatical'], sub['ungrammatical'])]
}
print('Synthetic eval suites:', list(syn_test_cases.keys()))

eval_df.sample(10)

## Train a Word-Level GPT (1 point)

In this part, you will train a Transformer language model on a small synthetic corpus
(`train.txt` / `dev.txt`) using a whitespace-split, word-level tokenizer. Below we first
define a set of **dataset/tokenizer-agnostic** utilities (tokenizer wrapper, batching,
model init, training loop, surprisal, minimal-pair eval). Section 2.2 will reuse exactly
the same utilities, swapping in BabyLM data and the pretrained GPT-2 tokenizer.


### Training Utilities & Helpers
Nothing to do here. You can hide after running.

In [ ]:
class TokenizerWrapper:
    """Build either from the synthetic word list,
    or from a huggingface pretrained tokenizer"""
    def __init__(self, encode_fn, decode_fn, vocab_size, bos_id, pad_id, eos_id=None,
                 contains_fn=None, name="tokenizer"):
        self._encode = encode_fn
        self._decode = decode_fn
        self.vocab_size = vocab_size
        self.bos_id = bos_id
        self.pad_id = pad_id
        self.eos_id = eos_id if eos_id is not None else bos_id
        # Optional membership check (used for OOV filtering on whitespace tokenizers).
        self._contains = contains_fn if contains_fn is not None else (lambda tok: True)
        self.name = name

    def encode(self, s):
        return self._encode(s)

    def decode(self, ids):
        return self._decode(ids)

    def contains(self, tok):
        """Does this token exist in the tokenizer?
        BPE tokenizers always return True (no OOV); whitespace ones may not."""
        return self._contains(tok)


def build_synthetic_tokenizer(train_text, pad_token='<PAD>', bos_token='<BOS>'):
    """Whitespace word-level tokenizer, identical to the original notebook.
    `train_text` is a list of strings (one per line)."""
    vocab = [pad_token, bos_token] + sorted(set(' '.join(train_text).split()))
    stoi = {ch: i for i, ch in enumerate(vocab)}
    itos = {i: ch for i, ch in enumerate(vocab)}

    def encode(s):
        # Filter OOV silently -- matches original sentence_surprisal behavior.
        return [stoi[t] for t in s.split() if t in stoi]

    def decode(ids):
        return ' '.join(itos[i] for i in ids)

    return TokenizerWrapper(
        encode_fn=encode,
        decode_fn=decode,
        vocab_size=len(vocab),
        bos_id=stoi[bos_token],
        pad_id=stoi[pad_token],
        eos_id=stoi[bos_token],   # synthetic vocab has no EOS; reuse BOS id
        contains_fn=lambda t: t in stoi,
        name='synthetic-word',
    )


def build_pretrained_tokenizer(name='gpt2'):
    """Wrap a HF pretrained tokenizer (GPT-2 by default).
    GPT-2 has no PAD by default, so we reuse EOS as PAD -- standard practice.
    Loss on PAD positions will be masked with -100, so the duplicate id is harmless."""
    hf_tok = GPT2TokenizerFast.from_pretrained(name)
    if hf_tok.pad_token is None:
        hf_tok.pad_token = hf_tok.eos_token

    def encode(s):
        return hf_tok.encode(s, add_special_tokens=False)

    def decode(ids):
        return hf_tok.decode(ids, skip_special_tokens=False)

    bos_id = hf_tok.bos_token_id if hf_tok.bos_token_id is not None else hf_tok.eos_token_id
    return TokenizerWrapper(
        encode_fn=encode,
        decode_fn=decode,
        vocab_size=hf_tok.vocab_size,
        bos_id=bos_id,
        pad_id=hf_tok.pad_token_id,
        eos_id=hf_tok.eos_token_id,
        contains_fn=lambda t: True,   # BPE has no OOV
        name=name,
    ), hf_tok


# ---- Data preparation -------------------------------------------------------

def tokenize_lines(lines, tokenizer, hf_tok=None, batch_tokenize=False):
    """Tokenize a list of strings into a list of 1D LongTensors (one per line).
    Empty / whitespace-only lines are skipped. If `hf_tok` is provided we use
    fast batched tokenization (much faster for large corpora like BabyLM)."""
    lines = [ln for ln in lines if ln and ln.strip()]
    if batch_tokenize and hf_tok is not None:
        encodings = hf_tok(lines, add_special_tokens=False)['input_ids']
        return [torch.tensor(ids, dtype=torch.long) for ids in encodings if len(ids) > 0]
    return [torch.tensor(tokenizer.encode(ln), dtype=torch.long)
            for ln in lines if len(tokenizer.encode(ln)) > 0]


# ---- Batching ---------------------------------------------------------------

def make_get_batch(train_data, dev_data, tokenizer, batch_size, block_size, device):
    """Returns a closure get_batch(split) -> (x, y).
    Mirrors the original logic: pick a random line, prepend BOS, then either
    pad to block_size (short lines) or random-crop (long lines, e.g. BabyLM).
    Labels mirror inputs but mask PAD positions with -100 (HF GPT2LMHeadModel
    will shift internally for next-token CE loss).
    """
    pad_id = tokenizer.pad_id
    bos_id = tokenizer.bos_id

    def get_batch(split):
        data_list = train_data if split == 'train' else dev_data
        batch_x, batch_y = [], []
        for _ in range(batch_size):
            idx = torch.randint(0, len(data_list), (1,)).item()
            tokens = [bos_id] + data_list[idx].tolist()

            if len(tokens) > block_size:
                # Random crop into longer documents; ensures we sometimes start mid-doc.
                start = torch.randint(0, len(tokens) - block_size + 1, (1,)).item()
                tokens = tokens[start:start + block_size]

            pad_len = block_size - len(tokens)
            x_item = torch.tensor(tokens + [pad_id] * pad_len, dtype=torch.long)
            y_item = torch.tensor(tokens + [-100] * pad_len, dtype=torch.long)
            batch_x.append(x_item)
            batch_y.append(y_item)

        x = torch.stack(batch_x).to(device)
        y = torch.stack(batch_y).to(device)
        return x, y

    return get_batch


# ---- Model init -------------------------------------------------------------

def build_model(vocab_size, block_size, n_embd, n_head, n_layer, dropout, device,
                eos_token_id=None):
    config = GPT2Config(
        vocab_size=vocab_size,
        n_positions=block_size,
        n_embd=n_embd,
        n_layer=n_layer,
        n_head=n_head,
        resid_pdrop=dropout,
        embd_pdrop=dropout,
        attn_pdrop=dropout,
        n_inner=4 * n_embd,
        bos_token_id=None,
        eos_token_id=eos_token_id if eos_token_id is not None else 0,
        initializer_range=0.01,
    )
    model = GPT2LMHeadModel(config).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model: {n_layer}L {n_head}H {n_embd}D | vocab={vocab_size} | block={block_size} | {n_params:.2f}M params")
    return model, config


# ---- Surprisal & minimal-pair eval -----------------------------------------

@torch.no_grad()
def sentence_surprisal(sentence, model, tokenizer, device, block_size, normalize=False):
    """Compute total (or mean) per-token surprisal of `sentence` under `model`.
    Prepends BOS to match training. Truncates to block_size if necessary."""
    model.eval()
    # Filter OOV -- relevant for the synthetic word tokenizer; no-op for BPE.
    filtered = [t for t in sentence.split() if tokenizer.contains(t)]
    if len(filtered) < 1:
        return float('inf')

    token_ids = [tokenizer.bos_id] + tokenizer.encode(' '.join(filtered))
    if len(token_ids) < 2:
        return float('inf')
    token_ids = token_ids[:block_size]
    tokens = torch.tensor([token_ids], dtype=torch.long, device=device)

    logits = model(input_ids=tokens).logits
    shift_logits = logits[:, :-1, :]
    shift_labels = tokens[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    total = -token_log_probs.sum().item()
    if normalize:
        total /= max(token_log_probs.shape[1], 1)
    return total


def evaluate_minimal_pairs(test_cases, model, tokenizer, device, block_size,
                           max_per_suite=None, verbose=True):
    """`test_cases`: dict[suite_name -> list of (good_sentence, bad_sentence) pairs].
    Returns (overall_acc, per_suite_acc_dict).
    `max_per_suite` caps pairs per suite (use during training to keep eval fast)."""
    model.eval()
    accs = {}
    for suite, pairs in test_cases.items():
        eval_pairs = pairs if max_per_suite is None else pairs[:max_per_suite]
        if not eval_pairs:
            continue
        correct = 0
        for s_good, s_bad in eval_pairs:
            sg = sentence_surprisal(s_good, model, tokenizer, device, block_size)
            sb = sentence_surprisal(s_bad, model, tokenizer, device, block_size)
            if sg < sb:
                correct += 1
        accs[suite] = correct / len(eval_pairs)

    overall = sum(accs.values()) / len(accs) if accs else 0.0
    if verbose:
        details = ' '.join(f'| {s}: {a:.2f}' for s, a in accs.items())
        print(f"Overall: {overall:.3f} {details}")
    model.train()
    return overall, accs


# ---- Validation loss --------------------------------------------------------

@torch.no_grad()
def estimate_loss(model, get_batch, eval_iters):
    model.eval()
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = get_batch('val')
        outputs = model(input_ids=X, labels=Y)
        losses[k] = outputs.loss.item()
    model.train()
    return losses.mean().item()




### Training Loop

In [ ]:
# TODO: Modify the training loop to return:
# - training PPL,
# - validation PPL, and
# - minimal pairs accuracy
# throughout training.

def train_model(model, get_batch, tokenizer, max_iters, eval_interval,
                train_loss_interval, eval_iters, learning_rate, block_size, device,
                test_cases=None, max_pairs_per_suite_during_train=None,
                weight_decay=0.1, warmup_frac=0.1, grad_clip=1.0):
    """Generic training loop. Reports val loss/PPL every `eval_interval` and,
    if `test_cases` is supplied, also runs minimal-pair eval there.
    TODO: Returns a dict of recorded metrics for plotting."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    warmup_iters = max(1, int(warmup_frac * max_iters))
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_iters)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, max_iters - warmup_iters), eta_min=learning_rate * 0.1)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[warmup_iters])
    train_losses_record = []
    start_time = time.time()
    model.to(device).train()

    for it in tqdm(range(max_iters)):
        # ---- Eval ----
        if it % eval_interval == 0 or it == max_iters - 1:
            val_loss = estimate_loss(model, get_batch, eval_iters)
            val_ppl = math.exp(min(val_loss, 20))   # guard early-iter overflow
            print(f"step {it:5d} | val loss {val_loss:.4f} | val ppl {val_ppl:.2f} "
                  f"| lr {scheduler.get_last_lr()[0]:.2e}")
            if test_cases is not None:
                overall, accs = evaluate_minimal_pairs(
                    test_cases, model, tokenizer, device, block_size,
                    max_per_suite=max_pairs_per_suite_during_train, verbose=True)
            model.train()

        # ---- Train step ----
        xb, yb = get_batch('train')
        outputs = model(input_ids=xb, labels=yb)
        loss = outputs.loss
        train_losses_record.append(loss.item())
        if it % train_loss_interval == 0 and it > 0:
            recent = train_losses_record[-train_loss_interval:]
            loss_mean = sum(recent) / len(recent)
            print(f"step {it:5d} | train loss {loss_mean:.4f} "
                  f"| train ppl {math.exp(min(loss_mean, 20)):.2f} "
                  f"| lr {scheduler.get_last_lr()[0]:.2e}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

    print(f"\nTraining complete in {time.time() - start_time:.1f}s")
    return # TODO


### Train Your Model

In [ ]:
# ----- Hyperparameters for synthetic training -----
syn_batch_size = 128
syn_block_size = 32
syn_max_iters = 1000
syn_eval_interval = 20
syn_train_loss_interval = 5
syn_learning_rate = 3e-3
syn_eval_iters = 50
syn_n_embd = 256
syn_n_head = 4
syn_n_layer = 6
syn_dropout = 0.1

In [ ]:
# ----- Load synthetic corpus and build word-level tokenizer -----
with open('train.txt') as f:
    syn_train_text = f.read().strip().split('\n')
with open('dev.txt') as f:
    syn_dev_text = f.read().strip().split('\n')

syn_tokenizer = build_synthetic_tokenizer(syn_train_text)
print(f"Vocabulary size: {syn_tokenizer.vocab_size}")
print(f"Train: {len(syn_train_text)} lines, {len(' '.join(syn_train_text).split())} tokens")
print(f"Dev:   {len(syn_dev_text)} lines, {len(' '.join(syn_dev_text).split())} tokens")

syn_train_data = tokenize_lines(syn_train_text, syn_tokenizer)
syn_dev_data   = tokenize_lines(syn_dev_text,   syn_tokenizer)

syn_get_batch = make_get_batch(
    syn_train_data, syn_dev_data, syn_tokenizer,
    batch_size=syn_batch_size, block_size=syn_block_size, device=device,
)


In [ ]:
# ----- Build synthetic model and train -----
syn_model, syn_config = build_model(
    vocab_size=syn_tokenizer.vocab_size,
    block_size=syn_block_size,
    n_embd=syn_n_embd, n_head=syn_n_head, n_layer=syn_n_layer,
    dropout=syn_dropout, device=device,
    eos_token_id=syn_tokenizer.eos_id,
)

syn_metrics = train_model(
    model=syn_model,
    get_batch=syn_get_batch,
    tokenizer=syn_tokenizer,
    max_iters=syn_max_iters,
    eval_interval=syn_eval_interval,
    train_loss_interval=syn_train_loss_interval,
    eval_iters=syn_eval_iters,
    learning_rate=syn_learning_rate,
    block_size=syn_block_size,
    device=device,
    test_cases=syn_test_cases,
    max_pairs_per_suite_during_train=None,   # synthetic eval is small; run all pairs
)


### Modify Logging Frequency

You will notice that changes in performance are very large in the first part of training, and slow down a lot later on. To better capture these patterns, modify the training loop above to perform evaluation at exponentially growing intervals (i.e., 1 step, 2 steps, 4 steps, etc.) until you reach 20% of `max_iters`. Retrain your model.

## Plot Results (1 point)

Create plots of training perplexity, validation perplexity, and minimal pairs accuracy throughout training (I recommend using seaborn's `relplot` function).

In [ ]:
import seaborn as sns

## Mini-Project 1 (3 points)

For this mini-project, extend your GPT-2 training experiments. Some suggestions:

- Perform a hyperparameter sweep and explore the tradeoff of training time (consider both training steps and wall clock time) and performance. Plot a Pareto curve or area under the curve (set the floor as the best loss across all runs).
- Write and evaluate on new test suites. Try to test for things that are hard for the model to learn (you'll notice accuracy on most test suites is very high early into training).
- Look for variability betweeen runs due to random seed. Do learning curves look consistent? Are the same examples difficult/easy?
- Experiment with tokenizers. Try the pretrained GPT-2 tokenizer, character tokenization, etc.
- Train small GPT-2s on the BabyLM Strict-Small corpus (Note: these runs will take at least 10-30 mins). Code is provided below to do this. Do one or more of the following:
  - Explore/optimize hyperparameters for performance/efficiency,
  - Analyze learning curves,
  - Experiment with tokenizers,
  - Write and eval on new test cases.

In [ ]:
# YOUR CODE

Describe your mini-project motivation, methods, and results.

[Your answer here]

## Generative AI Usage
Describe any contributions made by generative AI in this portion of the assignment.

[Your answer here]

## Train on BabyLM (Optional)

Now we reuse the utilities above to train on the
[BabyLM-2026 Strict-Small](https://huggingface.co/datasets/BabyLM-community/BabyLM-2026-Strict-Small)
corpus (~10M tokens, single `train` split with a `text` column) with the
[BabyLM-dev](https://huggingface.co/datasets/BabyLM-community/BabyLM-dev) set as validation,
tokenize with the pretrained **GPT-2** BPE tokenizer, and evaluate on
[BLiMP](https://huggingface.co/datasets/nyu-mll/blimp) (67 syntactic minimal-pair suites).

The dev split is published as raw `*.dev` text files (one per source domain), so we
download them with `huggingface_hub` rather than `load_dataset`.


In [ ]:
# ----- Load BabyLM Strict-Small (train) and BabyLM-dev (validation) -----
from huggingface_hub import hf_hub_download, list_repo_files

# Train: parquet-backed HF Dataset, single 'train' split, 'text' column.
babylm_train = load_dataset('BabyLM-community/BabyLM-2026-Strict-Small', split='train')
print('Train columns:', babylm_train.column_names)
print('Train rows:   ', len(babylm_train))

babylm_train_text = [t for t in babylm_train['text'] if t and t.strip()]
print(f'Train non-empty lines: {len(babylm_train_text):,}')

# Dev: raw *.dev files. List them and download each.
dev_files = [f for f in list_repo_files('BabyLM-community/BabyLM-dev', repo_type='dataset')
             if f.endswith('.dev')]
print('Dev files:', dev_files)

babylm_dev_text = []
for fname in dev_files:
    path = hf_hub_download(repo_id='BabyLM-community/BabyLM-dev',
                           filename=fname, repo_type='dataset')
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                babylm_dev_text.append(line)
print(f'Dev non-empty lines:   {len(babylm_dev_text):,}')


In [ ]:
def build_retrained_gpt2_tokenizer(texts, vocab_size=8000, batch_size=1000, name=None):
    """Take the GPT-2 tokenizer's config (byte-level BPE, <|endoftext|> special token,
    pretokenization rules) and learn fresh merges from `texts` at `vocab_size`."""
    base = GPT2TokenizerFast.from_pretrained('gpt2')

    # Generator: yields chunks instead of materializing 1.1M strings as one list arg.
    # Faster than one-string-at-a-time, lighter on RAM than a single giant list.
    def chunks():
        for i in range(0, len(texts), batch_size):
            yield texts[i:i + batch_size]

    hf_tok = base.train_new_from_iterator(
        text_iterator=chunks(),
        vocab_size=vocab_size,
        length=len(texts),          # only used for the progress bar
    )
    if hf_tok.pad_token is None:
        hf_tok.pad_token = hf_tok.eos_token

    def encode(s):
        return hf_tok.encode(s, add_special_tokens=False)
    def decode(ids):
        return hf_tok.decode(ids, skip_special_tokens=False)

    bos_id = hf_tok.bos_token_id if hf_tok.bos_token_id is not None else hf_tok.eos_token_id
    return TokenizerWrapper(
        encode_fn=encode, decode_fn=decode,
        vocab_size=hf_tok.vocab_size,
        bos_id=bos_id,
        pad_id=hf_tok.pad_token_id,
        eos_id=hf_tok.eos_token_id,
        contains_fn=lambda t: True,
        name=name or f'gpt2-retrained-{vocab_size}',
    ), hf_tok


babylm_tokenizer, babylm_hf_tok = build_retrained_gpt2_tokenizer(
    babylm_train_text, vocab_size=8000,
)
print(f'Trained tokenizer | vocab={babylm_tokenizer.vocab_size} '
      f'| bos={babylm_tokenizer.bos_id} pad={babylm_tokenizer.pad_id}')

# Quick sanity check on what it learned
for s in ["The quick brown fox.", "Once upon a time there was a little girl."]:
    ids = babylm_tokenizer.encode(s)
    print(f'  {len(ids)} tokens: {babylm_hf_tok.convert_ids_to_tokens(ids)}')

print(f"Tokenizer: {babylm_tokenizer.name} | vocab_size={babylm_tokenizer.vocab_size}")
print(f"  bos_id={babylm_tokenizer.bos_id}  pad_id={babylm_tokenizer.pad_id}  eos_id={babylm_tokenizer.eos_id}")


In [ ]:
# ----- Tokenize BabyLM (batched fast tokenization for speed) -----
# 1.1M short lines ~ 10M BPE tokens; this takes ~2min on Colab.
t0 = time.time()
babylm_train_data = tokenize_lines(
    babylm_train_text, babylm_tokenizer, hf_tok=babylm_hf_tok, batch_tokenize=True)
babylm_dev_data = tokenize_lines(
    babylm_dev_text, babylm_tokenizer, hf_tok=babylm_hf_tok, batch_tokenize=True)
print(f'Tokenized in {time.time()-t0:.1f}s')

n_train_toks = sum(t.numel() for t in babylm_train_data)
n_dev_toks   = sum(t.numel() for t in babylm_dev_data)
print(f'Train: {len(babylm_train_data):,} lines, {n_train_toks:,} BPE tokens')
print(f'Dev:   {len(babylm_dev_data):,} lines, {n_dev_toks:,} BPE tokens')


In [ ]:
# ----- Load BLiMP (67 minimal-pair suites) -----
# Each config is one suite of 1000 pairs with sentence_good / sentence_bad fields.
blimp_configs = get_dataset_config_names('nyu-mll/blimp')
print(f'BLiMP suites: {len(blimp_configs)}')

blimp_test_cases = {}
for cfg in tqdm(blimp_configs, desc='Loading BLiMP'):
    ds = load_dataset('nyu-mll/blimp', cfg, split='train')
    blimp_test_cases[cfg] = list(zip(ds['sentence_good'], ds['sentence_bad']))

total_pairs = sum(len(v) for v in blimp_test_cases.values())
print(f'Loaded {total_pairs:,} BLiMP pairs across {len(blimp_test_cases)} suites')


In [ ]:
# ----- Hyperparameters for BabyLM training -----
# A bigger context length (128) is useful here: BabyLM lines are longer than
# the synthetic ones, and BLiMP minimal pairs are full sentences.
# A smaller LR is appropriate because the GPT-2 vocab is ~50k vs ~1k synthetic.
bl_batch_size = 32
bl_block_size = 48
bl_max_iters = 10000
bl_eval_interval = 200
bl_train_loss_interval = 50
bl_learning_rate = 3e-4
bl_eval_iters = 50
bl_n_embd = 256
bl_n_head = 8
bl_n_layer = 8
bl_dropout = 0.1
bl_max_pairs_per_suite = 20

bl_get_batch = make_get_batch(
    babylm_train_data, babylm_dev_data, babylm_tokenizer,
    batch_size=bl_batch_size, block_size=bl_block_size, device=device,
)


In [ ]:
# ----- Build the BabyLM model (fresh, from-scratch GPT-2 config) -----
bl_model, bl_config = build_model(
    vocab_size=babylm_tokenizer.vocab_size,
    block_size=bl_block_size,
    n_embd=bl_n_embd, n_head=bl_n_head, n_layer=bl_n_layer,
    dropout=bl_dropout, device=device,
    eos_token_id=babylm_tokenizer.eos_id,
)


In [ ]:
# ----- Train on BabyLM with BLiMP eval at every eval_interval -----
# 67 suites x 1000 pairs x 2 forward passes is too slow to run every eval step,
# so we cap pairs per suite during training. A full per-suite eval follows below.
bl_metrics = train_model(
    model=bl_model,
    get_batch=bl_get_batch,
    tokenizer=babylm_tokenizer,
    max_iters=bl_max_iters,
    eval_interval=bl_eval_interval,
    train_loss_interval=bl_train_loss_interval,
    eval_iters=bl_eval_iters,
    learning_rate=bl_learning_rate,
    block_size=bl_block_size,
    device=device,
    test_cases=blimp_test_cases,
    max_pairs_per_suite_during_train=bl_max_pairs_per_suite,
)


In [ ]:
# ----- Final full BLiMP evaluation (all 1000 pairs per suite) -----
overall_acc, per_suite = evaluate_minimal_pairs(
    blimp_test_cases, bl_model, babylm_tokenizer, device,
    block_size=bl_block_size, max_per_suite=None, verbose=False,
)
print(f'BLiMP overall accuracy: {overall_acc:.4f}')
print()
for suite, acc in sorted(per_suite.items(), key=lambda kv: -kv[1]):
    print(f'  {acc:.3f}  {suite}')


# Section 3: Controlled Rearing and Syntactic Editing (7 points)

## Introduction to Stanza (0 points)

In controlled rearing studies, LM training corpora are modified according to some rules, and the effects on downstream tasks like acceptability judgments are observed. Parsers or other linguistic annotation tools are often used in the pipeline to formulate the rules for filtering or editing corpora.

[Stanza](https://stanfordnlp.github.io/stanza/) is a Python NLP library developed by the Stanford NLP Group. It provides a suite of performant tools for linguistic analysis, including:

- **Dependency Parsing** — constructing a graph in which each word is connected to the word it depends on syntactically
- **Constituency Parsing** - grouping words hierarchical into syntactic phrases
- **Part-of-Speech (POS) tagging** — labeling each word with its grammatical category
- **Lemmatization** — reducing words to their base form
- **Named Entity Recognition (NER)** — identifying entities like names, locations, etc.

Run the following code blocks to:
1. Install stanza
2. Download the models
3. Test stanza on some sentences

In [ ]:
!pip install stanza

In [ ]:
import stanza
stanza.download("en")
nlp = stanza.Pipeline(lang='en', processors='tokenize,mwt,pos,lemma,depparse')

In [ ]:
def doc_to_df(doc):
    sents = []
    target_columns = ["sent_id", "id", "text", "lemma", "upos", "head", "deprel", "xpos", "start_char", "end_char", "feats", "misc"]

    for i, sent in enumerate(doc.to_dict()):
        df_sent = pd.DataFrame([w for w in sent if type(w["id"])==int])
        df_sent["sent_id"] = i

        # Add any missing columns with NaN values
        for col in target_columns:
            if col not in df_sent.columns:
                df_sent[col] = None # Using None, pandas will convert to NaN for numeric types

        # Reorder and select the desired columns
        df_sent = df_sent[target_columns]
        sents.append(df_sent)
    return pd.concat(sents)

text = """The lemon (Citrus × limon) is a species of small evergreen tree in the Citrus genus of the flowering plant family Rutaceae. A true lemon is a hybrid of the citron and the bitter orange. Its origins are uncertain, but some evidence suggests lemons originated during the 1st millennium BC in what is now northeastern India. Some other citrus fruits are called lemon.
The yellow fruit of the lemon tree is used throughout the world, primarily for its juice. The pulp and rind are used in cooking and baking. The juice of the lemon is about 5–6% citric acid, giving it a sour taste. This makes it a key ingredient in drinks and foods such as lemonade and lemon meringue pie.
In 2024, world production was 23 million tonnes, led by India and Mexico with 31% of the total."""

doc = nlp(text)
doc_to_df(doc).head(50)

You can play around with Stanza above, or use the interactive demo on their website: https://stanza.stanford.edu/

## Filter Sentences with Determiner-Adjective-Noun (1.5 points)

Your first task with Stanza is to create a pipeline for syntactic filtering. Suppose we want to remove from our trainig corpus any sentences that contain noun with a determiner (e.g., "the", "a", "every") and an adjective as dependents.

Here are some examples:

- The red ball bounced. --> FILTER
- Buddy chased a fluffy squirrel. --> FILTER
- This super tall guy sat in front of me. --> FILTER
- Toby played with a girl with long pigtails. --> DO NOT FILTER
- The ball rolled down the hill quickly. --> DO NOT FILTER

**NOTE: Your filter might not always work if Stanza incorrectly parses a sentence.**

In [ ]:
def syntactic_filter(sentence):
    """Return True if and only if `sentence` contains a determiner, adjective,
    and noun in the same noun phrase.
    """
    # <YOUR CODE HERE>
    return False

test_cases = [
    ("The red ball bounced.", True),
    ("Buddy chased a fluffy squirrel.", True),
    ("This super tall guy sat in front of me.", True),
    ("Toby played with a girl with long pigtails.", False),
    ("The ball rolled down the hill quickly.", False),
    ("Nice guys finish last.", False)
]

for tc in test_cases:
    prediction = syntactic_filter(tc[0])
    if prediction == tc[1]:
        print(f"Correct | Sentence={tc[0]} | Prediction={prediction}")
    else:
        print(f"Incorrect | Sentence={tc[0]} | Prediction={prediction}")

In [ ]:
test_cases = [
    ("The dog barked.", "dog The barked."),
    ("Every dog chased a squirrel.", "dog Every chased squirrel a."),
    ("The dog chasing squirrels barked.", "dog chasing squirrels The barked."),
    ("The dog with a fluffy tail barked.", "dog with fluffy tail a the barked."),
    ("The president of Fluvlandia, a highly educated person, understands the economy.", "president of Fluvlandia, highly educated person a, The understands economy the."),
    ("Dogs like to chase squirrels.", "Dogs like to chase squirrels."),
]

No run `filter` over the first 1000 examples in the BabyLM Strict-Small corpus.

In [ ]:
babylm_train = load_dataset('BabyLM-community/BabyLM-2026-Strict-Small', split='train')
filtered = []
not_filtered = []
for sentence in babylm_train['text'][:1000]:
    if syntactic_filter(sentence):
        filtered.append(sentence)
    else:
        not_filtered.append(sentence)

print("=========== FILTERED ===========")
print(len(filtered))
for sentence in filtered:
    print(sentence)

print("=========== NOT FILTERED ===========")
print(len(not_filtered))
for sentence in not_filtered[:20]:
    print(sentence)

## Create a Counterfactual Corpus with Det-Noun Order (1.5 points)

Now use Stanza to write a function that will edit noun phrases to put the determiner at the end of the entire noun phrase.

Hints:

- Identify the noun phrase associated with a noun by looking recursively at all of its descendents.
- Don't worry about punctuation, capitalization, and appositives.

Examples:

- The dog barked. --> dog The barked.
- Every dog chased a squirrel. --> dog Every chased squirrel a.
- The dog chasing squirrels barked. --> dog chasing squirrels The barked.
- The dog with a fluffy tail barked. --> dog with fluffy tail a the barked.
- The president of Fluvlandia, a highly educated person, understands the economy. --> president of Fluvlandia, highly educated person a, The understands economy the.
- Dogs like to chase squirrels. --> Dogs like to chase squirrels.

**NOTE: Your `manipulate` function might not always work if Stanza incorrectly parses a sentence.**

In [ ]:
test_cases = [
    ("The dog barked.", "dog The barked."),
    ("Every dog chased a squirrel.", "dog Every chased squirrel a."),
    ("The dog chasing squirrels barked.", "dog chasing squirrels The barked."),
    ("The dog with a fluffy tail barked.", "dog with fluffy tail a The barked."),
    ("The president of Fluvlandia, a highly educated person, understands the economy.",
     "president of Fluvlandia, highly educated person a, The understands economy the."),
    ("Dogs like to chase squirrels.", None),
    ("Dogs with fluffy tails like to chase squirrels.", None),
]

In [ ]:
def manipulate(sentence):
    """Return a manipulated sentence with all determiners moved to the end
    of the noun phrase if possible. Return None if there are no alterations.
    """
    # <YOUR CODE HERE>
    return None

test_cases = [
    ("The dog barked.", "dog The barked."),
    ("Every dog chased a squirrel.", "dog Every chased squirrel a."),
    ("The dog chasing squirrels barked.", "dog chasing squirrels The barked."),
    ("The dog with a fluffy tail barked.", "dog with fluffy tail a The barked."),
    ("The president of Fluvlandia, a highly educated person, understands the economy.",
     "president of Fluvlandia, highly educated person a, The understands economy the."),
    ("Dogs like to chase squirrels.", None),
    ("Dogs with fluffy tails like to chase squirrels.", None),
]

for tc in test_cases:
    output = manipulate(tc[0])
    if output == tc[1]:
        print(f"Correct | Sentence={tc[0]} | Output={output}")
    else:
        print(f"Incorrect | Sentence={tc[0]} | Output={output}")

Now run your `manipulate` on the first 1000 examples from the BabyLM corpus.

In [ ]:
manipulated = []
not_manipulated = []
for sentence in babylm_train['text'][:1000]:
    maniuplated_sentence = manipulate(sentence)
    if maniuplated_sentence:
        manipulated.append(maniuplated_sentence)
    else:
        not_manipulated.append(sentence)

print("=========== MANIPULATED ===========")
print(len(manipulated))
for sentence in manipulated[:20]:
    print(sentence)

print("=========== NOT MANIPULATED ===========")
print(len(not_manipulated))
for sentence in not_manipulated[:20]:
    print(sentence)

## Mini-Project 2 (4 points)

For your second mini-project, perform a controlled rearing experiment with BabyLMs. Some suggested topics:


**Manipulate `PCFG.csv` in some way** and measure the effect on performance or learning trajectories. One advantage of this is that models are very quick to train, so you could do a large number of training runs and sweep through multiple values for your independent variable(s). However, highly synthetic experiments are less cognitively plausible.

  - Ablate or filter words or rules in certain contexts. Test for generalization.
  - Inject noise (ungrammatical examples) in various quantities. Test for robustness.
  - Edit the weights (probabilities) of different rules or lexical items. Plot the relationship between frequency and learnability.
  - Add subject-verb agreement to the grammar. Evaluate on agreement attraction errors.
  - Edit the grammar so that some verb heads come before their arguments and some come after. Observe the effect on learning trajectories.
  - Expand the vocabulary synthetically to include 10x and 100x the number of entries.
  - Add a second type of bracket to the grammar. Evaluate for learning of properly nested brackets (see the [Dyck language](https://en.wikipedia.org/wiki/Dyck_language)).

**Manipulate the BabyLM corpus** using `stanza`. This experiments are more cognitively plausible, but performing accurate syntactic manipulations may be technically challenging due to the complexity of the parsing framework or the accuracy of the parser, and the training times necessary to observe an effect of your manipulation(s) may be prohibitively long.

  - Run your `syntactic_filter` on the training data and measure the effect of filtering different amounts of data on relevant BLiMP test cases related to determiner-noun agreement.
  - Run your `manipulate` on the trainig data. Write minimal pairs that contrast the original and counterfactual word order. Measure the effect of manipulating different amounts of data on your test cases.
  - Conduct a corpus analysis on frequency of other syntactic phenomena.
  - Perform a novel syntactic manipulation. Validate the effectiveness and report precision and recall.


In [ ]:
# YOUR CODE

Describe your mini-project

[Your answer here]

## Generative AI Usage
Describe any contributions made by generative AI in this portion of the assignment.

[Your answer here]

# Section 4: Convert to .pdf and Submit

This section will walk you through instructions to convert your notebook to .pdf.

- Step 0: Ensure your notebook's outputs are displayed properly.
  - Make sure all cells were run in order and their outputs are as intended.
  - Clear any outputs from cells that don't need to be assessed.
- Step 1: Download your notebook as an `.ipynb` file.
  - Go to `File > Download > Download .ipynb`.
- Step 2: Drag and drop your `.ipynb` file into the `contents` directory of your notebook. You can find this by clicking on the folder icon on the left-hand side of the colab notebook. Ensure the filename in the `!jupyter nbconvert` command matches the uploaded file.
- Step 3: Run the codeblock below.
- Step 4: Download the output .pdf from the `content` directory. It should look like a nicely formatted LaTeX document.
  - Common error: In markdown files, ensure that all bulleted or numbered lists have an empty line above them (unless they are the first line in the markdown file). Otherwise, they will not render properly and **we may deduct points**.
- Step 5: Upload your assignment to gradescope.


In [ ]:
# Install the necessary LaTeX packages for PDF conversion. (This will take a minute or two.)
!apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic

# Convert your notebook to PDF. Make sure the notebook name is correct.
!jupyter nbconvert --to pdf /content/DSC_291_HW1_Part_2_Training_BabyLMs.ipynb

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
texlive-fonts-recommended is already the newest version (2021.20220204-1).
texlive-plain-generic is already the newest version (2021.20220204-1).
texlive-xetex is already the newest version (2021.20220204-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
[NbConvertApp] Converting notebook /content/DSC_291_HW1_Part_2_Training_BabyLMs.ipynb to pdf
[NbConvertApp] Writing 162918 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 154323 bytes to /content/DSC_291_HW1_Part_2_Training_BabyLMs.pdf
